# Lab: an autoscaling simulator

Run a discrete-event simulator of an LLM fleet with realistic request lengths, cold starts, a daily ramp and a sudden spike. Compare static, reactive and scheduled policies, sweep cold start time, remove the spike, change the variance, and write a policy recommendation backed by numbers.

This notebook holds the code from the lesson **[Lab: an autoscaling simulator](https://inferenceacademy.com/learn/prod-scaling/lab-autoscaling-simulator)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

No GPU needed: the default CPU runtime is enough.

## What the simulator models

## Step 1: save and run the simulator

In [ ]:
%%writefile scalesim.py
"""Discrete-event simulator of an LLM serving fleet with an autoscaler. Standard library only."""
import heapq, math, random, statistics, sys
from dataclasses import dataclass

SLOTS = 8                 # concurrent generations per replica
MEAN_SERVICE = 8.0        # seconds per request (prefill + decode), lognormal
SIGMA = 0.8               # lognormal shape (heavier tail = bigger sigma)
COLD_START = 180          # seconds from scale-out decision to ready replica
DAY = 4 * 3600            # simulated horizon: a 4-hour "day" (compressed)
QUEUE_CAP = 200           # reject beyond this many waiting
SLO_WAIT = 2.0            # seconds of queue wait we consider acceptable

def rate_at(t):           # requests per second: a slow ramp, a peak, and a sudden spike
    base = 3 + 7 * math.sin(math.pi * min(t, DAY) / DAY) ** 2       # 3 -> 10 -> 3 rps
    spike = 8 if 7200 <= t < 7500 else 0                           # +8 rps for 5 minutes
    return base + spike

def lognormal(mean, sigma, rng):
    mu = math.log(mean) - sigma * sigma / 2
    return rng.lognormvariate(mu, sigma)

@dataclass
class Result:
    name: str; p50: float; p95: float; p99: float; slo_miss: float; dropped: int; served: int; replica_hours: float; max_replicas: int

def simulate(policy, seed=1):
    rng = random.Random(seed)
    # events: (time, seq, kind, payload)
    ev, seq = [], 0
    def push(t, kind, payload=None):
        nonlocal seq; seq += 1; heapq.heappush(ev, (t, seq, kind, payload))
    ready = policy.initial; pending = 0          # ready replicas, replicas starting
    in_flight = 0; queue = []                    # queue of arrival times
    waits = []; dropped = 0; served = 0
    replica_seconds = 0.0; last_t = 0.0; max_rep = ready
    # arrivals via thinning of a non-homogeneous Poisson process
    lam_max = 25.0; t = 0.0
    while t < DAY:
        t += rng.expovariate(lam_max)
        if t < DAY and rng.random() < rate_at(t) / lam_max: push(t, "arrive")
    for k in range(0, DAY, policy.interval): push(float(k), "tick")
    scale_in_ok_since = None
    def try_start(now):
        nonlocal in_flight
        while queue and in_flight < ready * SLOTS:
            a = queue.pop(0); waits.append(now - a); in_flight += 1
            push(now + lognormal(MEAN_SERVICE, SIGMA, rng), "done")
    while ev:
        now, _, kind, payload = heapq.heappop(ev)
        replica_seconds += (ready + pending) * (now - last_t); last_t = now
        if kind == "arrive":
            if len(queue) >= QUEUE_CAP: dropped += 1
            else: queue.append(now); try_start(now)
        elif kind == "done":
            in_flight -= 1; served += 1; try_start(now)
        elif kind == "up":
            pending -= 1; ready += 1; max_rep = max(max_rep, ready); try_start(now)
        elif kind == "tick":
            desired = policy.desired(now, ready + pending, in_flight, len(queue))
            total = ready + pending
            if desired > total:
                for _ in range(desired - total): pending += 1; push(now + COLD_START, "up")
                scale_in_ok_since = None
            elif desired < ready:
                # scale in only after a stabilisation window of continuous "too many"
                if scale_in_ok_since is None: scale_in_ok_since = now
                if now - scale_in_ok_since >= policy.scale_in_window:
                    ready = max(desired, ready - max(1, ready // 10))   # shed up to 10% per step (drain assumed instant)
                    scale_in_ok_since = now
            else:
                scale_in_ok_since = None
    waits.sort()
    q = lambda p: waits[min(len(waits) - 1, int(p * len(waits)))] if waits else 0.0
    miss = sum(1 for w in waits if w > SLO_WAIT) / max(1, len(waits) + dropped)
    return Result(policy.name, q(.5), q(.95), q(.99), miss + dropped / max(1, len(waits) + dropped), dropped, served, replica_seconds / 3600, max_rep)

class Static:
    def __init__(self, n): self.name = f"static-{n}"; self.initial = n; self.interval = 15; self.scale_in_window = 10**9; self.n = n
    def desired(self, now, total, in_flight, waiting): return self.n

class Reactive:
    """Scale on in-flight + waiting requests per replica against a target fraction of capacity."""
    def __init__(self, target_util, min_r=2, max_r=30, scale_in_window=600, name=None):
        self.name = name or f"reactive-{int(target_util*100)}%"; self.initial = min_r; self.interval = 15
        self.target = target_util; self.min_r = min_r; self.max_r = max_r; self.scale_in_window = scale_in_window
    def desired(self, now, total, in_flight, waiting):
        need = math.ceil((in_flight + waiting) / (SLOTS * self.target))
        return max(self.min_r, min(self.max_r, need))

class Scheduled(Reactive):
    """Reactive plus a schedule that raises the floor before the known daily peak."""
    def __init__(self, target_util, floor_fn, **kw):
        super().__init__(target_util, name=f"scheduled+reactive-{int(target_util*100)}%", **kw); self.floor_fn = floor_fn
    def desired(self, now, total, in_flight, waiting):
        return max(super().desired(now, total, in_flight, waiting), self.floor_fn(now + COLD_START))

def floor_for_rate(t):          # forecast floor: replicas needed for the expected rate at t at 70% utilisation
    need = rate_at(t) * MEAN_SERVICE / (SLOTS * 0.7)
    return math.ceil(need)

def policies():
    return [Static(8), Static(14), Static(22),
            Reactive(0.85), Reactive(0.65), Reactive(0.5),
            Scheduled(0.65, floor_for_rate)]

if __name__ == "__main__":
    seeds = range(1, int(sys.argv[1]) + 1) if len(sys.argv) > 1 else range(1, 6)
    print(f"{'policy':28s} {'p95 wait':>9s} {'p99 wait':>9s} {'SLO miss':>9s} {'dropped':>8s} {'replica-h':>10s} {'peak':>5s} {'cost/1k req':>12s}")
    for i in range(len(policies())):
        rs = [simulate(policies()[i], seed=s) for s in seeds]
        m = lambda f: statistics.mean(f(r) for r in rs)
        rh = m(lambda r: r.replica_hours); sv = m(lambda r: r.served)
        print(f"{rs[0].name:28s} {m(lambda r: r.p95):9.2f} {m(lambda r: r.p99):9.2f} {m(lambda r: r.slo_miss)*100:8.1f}% {m(lambda r: r.dropped):8.0f} {rh:10.1f} {max(r.max_replicas for r in rs):5d} {rh/sv*1000:12.3f}")

In [ ]:
!python3 scalesim.py 5

Output shown in the lesson (yours will differ a little):

```text
policy                        p95 wait  p99 wait  SLO miss  dropped  replica-h  peak  cost/1k req
static-8                         26.35     27.70     55.8%     8968       32.1     8        0.370
static-14                         0.59     14.68      5.8%     1007       56.2    14        0.593
static-22                         0.00      0.00      0.0%        0       88.1    22        0.920
reactive-85%                      0.00     13.77      4.4%      433       80.6    30        0.845
reactive-65%                      0.00      0.93      0.7%       12       83.5    30        0.872
reactive-50%                      0.00      0.00      0.6%       12       88.2    30        0.921
scheduled+reactive-65%            0.00      0.00      0.6%        4       83.8    30        0.875
```

## Step 2: understand the code

## Step 3: experiments

### Experiment A: cold start sweep

### Experiment B: remove the spike

### Experiment C: heavier-tailed request lengths

### Experiment D: how big must a static fleet be?

### Experiment E: forecast quality

### Experiment F: scale-in aggressiveness

## Step 4: design your own policy

## Step 5: calibrate to your service (optional but valuable)

## Deliverable: a one-page recommendation

## Checklist

## Troubleshooting

## Stretch goals

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/prod-scaling/lab-autoscaling-simulator